#LanceDB - a vector database for LLM application

In [1]:
import lancedb

db= lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database')

In [2]:
db.uri

'c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database'

In [5]:
import json

with open ("animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())
    
data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [11]:
data[0].values()

dict_values(['A small brown dog running.', [0.12, 0.85, 0.33]])

In [13]:
table_animals = db.create_table("animals_text", exist_ok=True, data = data, mode = "overwrite")
table_animals

LanceTable(name='animals_text', version=2, _conn=LanceDBConnection(uri='c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database'))

In [14]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [16]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
    
]

table_animals.add(more_data)

AddResult(version=4)

In [18]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table and then delete it

In [22]:
from lancedb.pydantic import LanceModel

# A Pydantic Model base class that can be converted to a LanceDB Table.
LanceModel

class JokeSchema(LanceModel):
    joke: str
    rating: str
    
db.create_table(name = "jokes", schema = JokeSchema, exist_ok=True)
db

LanceDBConnection(uri='c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database')

In [23]:
db.table_names()

['animals_text', 'jokes']

In [ ]:
db.drop_table("jokes")

In [27]:
db.table_names()

['animals_text']

## Oen existing table

In [28]:
db.open_table("animals_text").head(2)

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1]]]

## Vector search in LanceDB

In [29]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [31]:
query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413


## Embedding API

- Idea: Want to put in text -> and it will automatically generate vector embeddings
- Put in a query and it will automatically generate vector embeddings
- Calculate closest distances

In [44]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name = "gemini-embedding001")
model

GeminiText(max_retries=7, name='gemini-embedding001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [52]:
import numpy as np
from dotenv import load_dotenv

load_dotenv()

hello_embedding = np.array(model.generate_embeddings("hello"))
hello_embedding.shape

(5, 3072)

In [53]:
model.ndims()

768

In [54]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name = "gemini-embedding-001")

class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(3072) = model.VectorField() # vector is destination of computed embedding

table_jokes = db.create_table("jokes", schema = JokeModel, exist_ok=True)
table_jokes

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database'))

In [63]:
import json
import pandas as pd

with open("jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


## Add data to table

In [65]:
table_jokes.add(df_jokes)

AddResult(version=3)

In [ ]:
table_jokes.head()

LanceTable(name='jokes', version=3, _conn=LanceDBConnection(uri='c:\\Users\\eriku\\Desktop\\DataenginerSTI2024-2026\\Github\\AIgeneering_four_week_course\\09_lancedb_vector_database\\vector_database'))

In [69]:
table_jokes.to_pandas().head()["vector"][0]

array([-0.02400176,  0.01247358, -0.02414474, ...,  0.01160614,
        0.00044886,  0.01217596], shape=(3072,), dtype=float32)

In [70]:
table_jokes.search("data egineering jokes").limit(8).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.480915
1,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.480915
2,"Data engineer motto: If it works, don’t touch ...","[-0.020296954, 0.020327171, -0.009069326, -0.0...",0.568333
3,"Data engineer motto: If it works, don’t touch ...","[-0.020296954, 0.020327171, -0.009069326, -0.0...",0.568333
4,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.653330
5,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.653330
6,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.662414
7,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.662414


In [77]:
table_jokes.search("water related").limit(8).to_pandas()

,joke,vector,_distance
0,I put “root beer” in a square glass… now it’s ...,"[-0.0060451194, 0.009575239, -0.017435549, -0....",0.799525
1,I put “root beer” in a square glass… now it’s ...,"[-0.0060451194, 0.009575239, -0.017435549, -0....",0.799525
2,I asked the data lake if it had my file. It sa...,"[-0.026169129, 0.01796462, -0.013938847, -0.09...",0.804208
3,I asked the data lake if it had my file. It sa...,"[-0.026169129, 0.01796462, -0.013938847, -0.09...",0.804208
4,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.826400
5,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.826400
6,"My pipeline failed again, so I’m logging out—l...","[-0.0013645191, 0.022650607, -0.039178334, -0....",0.828014
7,"My pipeline failed again, so I’m logging out—l...","[-0.0013645191, 0.022650607, -0.039178334, -0....",0.828014


## Hybrid search

Combines traditional keyword-based search with vector similaritysearch

In [79]:
# To enable keyword search, we need to create an index on the joke column
table_jokes.create_fts_index("joke", replace=True)

In [84]:
from lancedb import rerankers

# reranks the results from a vector and full text search
reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "water related",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="joke",
).rerank(reranker).limit(8).to_pandas()

results

,joke,vector,_relevance_score
0,I put “root beer” in a square glass… now it’s ...,"[-0.0060451194, 0.009575239, -0.017435549, -0....",0.016393
1,I put “root beer” in a square glass… now it’s ...,"[-0.0060451194, 0.009575239, -0.017435549, -0....",0.016129
2,I asked the data lake if it had my file. It sa...,"[-0.026169129, 0.01796462, -0.013938847, -0.09...",0.015873
3,I asked the data lake if it had my file. It sa...,"[-0.026169129, 0.01796462, -0.013938847, -0.09...",0.015625
4,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.015385
5,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.015152
6,"My pipeline failed again, so I’m logging out—l...","[-0.0013645191, 0.022650607, -0.039178334, -0....",0.014925
7,"My pipeline failed again, so I’m logging out—l...","[-0.0013645191, 0.022650607, -0.039178334, -0....",0.014706


## Rule of thumb

- For exact matching -> FTS
- Meaning based matching -> vector search
- Both, unpredicatble or mixed querires -> hybrid search
